# 🌸 Nayari Dataset Builder  v3
**Run locally.** Scans `dataset/` (including subdirectories) for `.md`, `.txt`, `.pdf`, and `.json` files, applies a robust multi-format parser, deduplicates, quality-filters, and exports `nayari_dataset.json`.  Then auto-uploads to Kaggle.

| Feature | Details |
|---|---|
| Speaker formats | `Name:`, `**Name**:`, `> Name:`, `[Name]:` |
| Scene splitting | Blank-line gaps, `---END---`, `<end>`, `===` dividers |
| Auto-aliases | Unknown speakers scanned from filenames & content |
| Quality filter | Min 2 turns, min 10 chars/message |
| Deduplication | MD5 fingerprint on normalised content |
| OOC stripping | `[OOC: …]` and `(OOC …)` blocks removed |
| Source tagging | Every conversation records its origin file |
| Token estimate | Rough GPT-style token count in the stats cell |


## 📦 Step 0 — Install & Import

In [1]:
%pip install pdfplumber kaggle -q

import re, json, os, shutil, subprocess, sys, hashlib, warnings
import pdfplumber
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore", message="Could not get FontBBox")

DATASET_DIR = Path("dataset")
OUTPUT_FILE = Path("nayari_dataset.json")

# ── pretty directory scan ────────────────────────────────────────────────────
all_files = sorted(
    f for f in DATASET_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in {".md", ".txt", ".pdf", ".json"}
)
print(f"Found {len(all_files)} source file(s) in dataset/:\n")
for f in all_files:
    rel = f.relative_to(DATASET_DIR)
    print(f"  [{f.suffix.upper():5}] {rel}  ({f.stat().st_size/1024:.1f} KB)")


Note: you may need to restart the kernel to use updated packages.
Found 196 source file(s) in dataset/:

  [.MD  ] set 1\md files\chats\Nayari_chat_1.md  (4.8 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_10.md  (4.8 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_11.md  (4.7 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_12.md  (4.6 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_13.md  (4.5 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_14.md  (4.7 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_15.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_16.md  (1.1 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_17.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_18.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_19.md  (1.4 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_2.md  (4.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_20.md  (1.0 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_21.md  (1.2 KB)
  [.MD  ] set 1\md files\chats\Nayari_chat_22.md  (1.2 KB)
  [.MD  ] se

## 1 — Character Details

In [2]:
# Locate the details file anywhere inside dataset/
details_candidates = list(DATASET_DIR.rglob("*details*"))
if not details_candidates:
    print("⚠️  No details file found — character metadata will be empty.")
    char_name = char_type = char_gender = char_traits = char_personality = ""
else:
    details_text = details_candidates[0].read_text(encoding="utf-8", errors="replace")

    def extract_field(text, field):
        m = re.search(rf"\*\*{field}\*\*:?\s*(.+)", text)
        return m.group(1).strip() if m else ""

    char_name        = extract_field(details_text, "Name")
    char_type        = extract_field(details_text, "Type")
    char_gender      = extract_field(details_text, "Gender")
    char_traits      = extract_field(details_text, "Traits")
    char_personality = extract_field(details_text, "Personally")
    print(f"Character : {char_name} | {char_type} | {char_gender}")
    print(f"Details file: {details_candidates[0].relative_to(DATASET_DIR)}")

lore_sections = []


Character : Nayari | Kemonomimi | Female (Cis)
Details file: set 1\md files\lore\Nayari_Details.md


## 2 — Robust Multi-Format Parser

Handles every speaker format seen in the repo:

```
Name: text           ← plain
**Name**: text       ← bold markdown
> Name: text         ← blockquote
[Name]: text         ← bracket
```

Scene splitting triggers on: `--- END ---`, `=== break ===`, `<end>`, or **3+ blank lines**.
OOC annotations `[OOC: …]` and `(OOC …)` are stripped from message content.

In [1]:
import json
import re
import time
import hashlib
from pathlib import Path
from collections import defaultdict

# --- 1. CONFIGURATION & REGEX (Your Logic) ---
BASE_USER_ALIASES      = {"me", "you", "tiaya"}
BASE_ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}

SPEAKER_RE = re.compile(
    r"""^(?:\*{1,2}|\[|>\s*)?([A-Za-z][A-Za-z0-9 _\'\-]*) (?:\*{1,2}|\])? :\s*(.*)$""",
    re.VERBOSE | re.MULTILINE
)
END_RE = re.compile(r"^[-=*]{3,}\s*(<?(end|break|scene)>?)?\s*[-=*]{0,}$", re.I)
OOC_RE = re.compile(r"\[OOC:?[^\]]*\]|\(OOC[^)]*\)", re.IGNORECASE)

class Col:
    BLUE = "\033[94m"; CYAN = "\033[96m"; GREEN = "\033[92m"
    YELLOW = "\033[93m"; RED = "\033[91m"; BOLD = "\033[1m"; END = "\033[0m"

# --- 2. THE CORE FUNCTIONS (Your Parser Logic) ---

def _strip_ooc(text: str) -> str:
    return OOC_RE.sub("", text).strip()

def _inject_scene_ends(text: str) -> str:
    return re.sub(r"(\r?\n){3,}", "\n--- END ---\n", text)

def _detect_aliases(text: str):
    counts = defaultdict(int)
    for line in text.splitlines():
        m = SPEAKER_RE.match(line.strip())
        if m: counts[m.group(1).strip().lower()] += 1
    u_extra, a_extra = set(), set()
    asst_keywords = {"nayari", "nayri", "aura", "goddess"}
    for name, cnt in counts.items():
        if cnt < 2: continue
        if any(kw in name for kw in asst_keywords): a_extra.add(name)
        else: u_extra.add(name)
    return u_extra, a_extra

def parse_chat_text(text: str, filename: str = ""):
    u_extra, a_extra = _detect_aliases(text)
    user_aliases = BASE_USER_ALIASES | u_extra
    asst_aliases = BASE_ASSISTANT_ALIASES | a_extra
    
    text = _inject_scene_ends(text)
    lines, conversations, current_messages = text.splitlines(), [], []
    current_role, buf = None, []

    def flush():
        nonlocal buf
        if current_role and buf:
            content = _strip_ooc(" ".join(" ".join(buf).split()).strip())
            if len(content) >= 10:
                current_messages.append({"role": current_role, "content": content})
        buf = []

    def save():
        if len(current_messages) >= 2:
            conversations.append({"messages": list(current_messages), "source": filename})
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s: continue
        if (s.startswith(("#", "-", "=")) and not SPEAKER_RE.match(s)) or END_RE.match(s):
            flush(); save(); current_role = None; continue
        m = SPEAKER_RE.match(s)
        if m:
            sp, rest = m.group(1).strip().lower(), m.group(2).strip()
            if sp in user_aliases: flush(); current_role = "user"; buf = [rest]
            elif sp in asst_aliases: flush(); current_role = "assistant"; buf = [rest]
            else: 
                if current_role: buf.append(s)
        elif current_role: buf.append(s)
    flush(); save()
    return conversations

# --- 3. THE EXECUTION LOOP (The Parser) ---

processed_data = {"conversations": [], "lore": [], "failed": []}
start_time = time.time()

# Ensure all_files exists (Discovery)
DATASET_DIR = Path("dataset") # Adjust this path as needed
all_files = sorted([f for f in DATASET_DIR.rglob("*") if f.suffix.lower() in {'.json', '.md', '.txt', '.pdf'} and f.is_file()])

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    # Truncate long paths for the UI
    display_name = (rel[:37] + '..') if len(rel) > 40 else rel
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx:03}/{len(all_files)}]{Col.END}"
    
    try:
        # JSON PROCESSING
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            # Logic: is it lore?
            if "lore" in rel.lower() or "world" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "content": data})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {display_name:<40} | {Col.YELLOW}Data Tree{Col.END}")
            else:
                convs = []
                if isinstance(data, list): convs = [{"messages": c["messages"], "source": rel} for c in data if "messages" in c]
                elif isinstance(data, dict):
                    if "conversations" in data: convs = data["conversations"]
                    elif "messages" in data: convs = [data]
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {display_name:<40} | {Col.CYAN}{len(convs)} convs{Col.END}")

        # TEXT / MD PROCESSING
        elif ext in {".txt", ".md"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            # Logic: is it lore or chat?
            if "lore" in rel.lower() or "details" in rel.lower():
                processed_data["lore"].append({"source": rel, "text": raw_text})
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}{len(raw_text.split()):,} words{Col.END}")
            else:
                convs = parse_chat_text(raw_text, rel)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")

        # PDF PROCESSING
        elif ext == ".pdf":
            import pdfplumber
            with pdfplumber.open(path) as pdf:
                text = "\n".join([p.extract_text() or "" for p in pdf.pages])
            convs = parse_chat_text(text, rel)
            if convs:
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📘 {Col.CYAN}PDF-CHAT{Col.END}   | {display_name:<40} | {Col.CYAN}{len(convs)} scenes{Col.END}")
            else:
                processed_data["lore"].append({"source": rel, "text": text})
                print(f"{prefix} 📘 {Col.YELLOW}PDF-LORE{Col.END}   | {display_name:<40} | {Col.YELLOW}Knowledge{Col.END}")

    except Exception as exc:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {display_name:<40} | {Col.RED}{type(exc).__name__}{Col.END}")
        processed_data["failed"].append(rel)

# --- 4. FINAL SMART BREAKDOWN ---

total_convs = len(processed_data['conversations'])
total_msgs = sum(len(c.get("messages", [])) for c in processed_data["conversations"])
total_lore = len(processed_data['lore'])
elapsed = time.time() - start_time

print(f"\n{Col.BOLD}{'═'*75}{Col.END}")
print(f"{Col.BOLD}📊 INTELLIGENCE ENGINE: FINAL DATASET BREAKDOWN{Col.END}")
print(f"{Col.BOLD}{'═'*75}{Col.END}")
print(f"⏱️  Processing Time:   {Col.BOLD}{elapsed:.2f} seconds{Col.END}")
print(f"💬 Total Conversations: {Col.GREEN}{total_convs:,}{Col.END}")
print(f"💬 Total Messages:      {Col.GREEN}{total_msgs:,}{Col.END}")
print(f"📜 Lore/Knowledge:      {Col.YELLOW}{total_lore:,} entries{Col.END}")
print(f"❌ Failed Files:        {Col.RED}{len(processed_data['failed'])}{Col.END}")

if total_convs > 0:
    avg = total_msgs / total_convs
    print(f"📈 Content Density:     {Col.CYAN}{avg:.1f} messages per conversation{Col.END}")

print(f"{Col.BOLD}{'═'*75}{Col.END}")


🧠 Intelligence Engine: Analyzing 196 files...

[001/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_1.md    | 1 scenes
[002/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_10.md   | 4 scenes
[003/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_11.md   | 4 scenes
[004/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_12.md   | 4 scenes
[005/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_13.md   | 4 scenes
[006/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_14.md   | 4 scenes
[007/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_15.md   | 1 scenes
[008/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_16.md   | 1 scenes
[009/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_17.md   | 1 scenes
[010/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_18.md   | 1 scenes
[011/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_19.md   | 1 scenes
[012/196] 📝 TXT-CHAT   | set 1\md files\chats\Nayari_chat_2.md    | 1 scenes
[013/196] 📝 TXT-CHAT   | set

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


[129/196] 📘 PDF-LORE   | set 1\pdf files\lore\Discovery .pdf      | Knowledge
[130/196] 📘 PDF-LORE   | set 1\pdf files\lore\Her Beliefs .pdf    | Knowledge
[131/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy201.pdf        | Knowledge
[132/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy202.pdf        | Knowledge
[133/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy203.pdf        | Knowledge
[134/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy204.pdf        | Knowledge
[135/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy205.pdf        | Knowledge
[136/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy206.pdf        | Knowledge
[137/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy207.pdf        | Knowledge
[138/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy208.pdf        | Knowledge
[139/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy209.pdf        | Knowledge
[140/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy210.pdf        | Knowledge
[141/196] 📘 PDF-LORE   | set 1\pdf files\other\kegy211.pdf      

## 3 — Parse All Source Files (.md / .txt / .pdf / .json)

In [2]:
import json
import time
from pathlib import Path

# --- ANSI Colors for "Smart" Console ---
class Col:
    BLUE = "\033[94m"
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    END = "\033[0m"

# Data storage
processed_data = {
    "conversations": [],
    "lore": [],
    "failed": []
}

def identify_content_type(text, file_path):
    """Advanced Heuristic: Checks for dialogue patterns vs descriptive prose."""
    if is_lore_file(file_path):
        return "lore"
        
    lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 2]
    if not lines: return "unknown"
    
    # Check for dialogue patterns: "Name:", "[Name]", "<Name>", or "- Name:"
    chat_indicators = 0
    sample_size = min(len(lines), 30)
    sample_lines = lines[:sample_size]
    
    for line in sample_lines:
        # Check for: "Speaker: ...", "[Speaker] ...", or "*Speaker*: ..."
        if any([":" in line[:25], 
                (line.startswith("[") and "]" in line[:25]),
                (line.startswith("*") and ":" in line[:30])]):
            chat_indicators += 1
            
    ratio = chat_indicators / sample_size
    return "chat" if ratio > 0.25 else "lore"

def get_stats_string(content_type, count, unit="items"):
    """Returns a formatted string for the live logger."""
    color = Col.CYAN if content_type == "chat" else Col.YELLOW
    return f"{color}{count} {unit}{Col.END}"

print(f"\n{Col.BOLD}{Col.BLUE}🧠 Intelligence Engine: Analyzing {len(all_files)} files...{Col.END}\n")
start_time = time.time()

for idx, path in enumerate(all_files, 1):
    rel = str(path.relative_to(DATASET_DIR))
    ext = path.suffix.lower()
    prefix = f"{Col.BOLD}[{idx}/{len(all_files)}]{Col.END}"
    
    try:
        # --- 1. JSON PROCESSING ---
        if ext == ".json":
            data = json.loads(path.read_text(encoding="utf-8", errors="replace"))
            
            # Smart Detection for Worldbooks
            is_worldbook = any(k in str(data).lower() for k in ["entries", "worldbook", "character_book"])
            
            if is_lore_file(path) or is_worldbook:
                processed_data["lore"].append({"source": rel, "content": data, "type": "json_lore"})
                print(f"{prefix} 📜 {Col.YELLOW}LORE-JSON{Col.END}  | {rel[:40]:<40} | {Col.YELLOW}Key-Value pairs detected{Col.END}")
            
            else:
                convs = []
                if isinstance(data, list): convs = [c for c in data if "messages" in c or "items" in c]
                elif isinstance(data, dict):
                    if "conversations" in data: convs = data["conversations"]
                    elif "messages" in data: convs = [data]
                
                for c in convs: c["source"] = rel
                processed_data["conversations"].extend(convs)
                msg_count = sum(len(c.get("messages", [])) for c in convs)
                print(f"{prefix} 💬 {Col.CYAN}CHAT-JSON{Col.END}  | {rel[:40]:<40} | {get_stats_string('chat', msg_count, 'msgs')}")

        # --- 2. PDF PROCESSING ---
        elif ext == ".pdf":
            # (Assuming extract_pdf is defined elsewhere)
            if is_lore_file(path):
                # Logic for forced lore extraction...
                processed_data["lore"].append({"source": rel, "type": "pdf_forced_lore"})
                print(f"{prefix} 📘 {Col.YELLOW}PDF-LORE{Col.END}   | {rel[:40]:<40} | {Col.YELLOW}Extracted via OCR/Text{Col.END}")
            else:
                chat_convs, lore_chunks = extract_pdf(path)
                if len(chat_convs) > len(lore_chunks):
                    processed_data["conversations"].extend(chat_convs)
                    print(f"{prefix} 📘 {Col.CYAN}PDF-CHAT{Col.END}   | {rel[:40]:<40} | {get_stats_string('chat', len(chat_convs), 'convs')}")
                else:
                    processed_data["lore"].append({"source": rel, "text": "\n".join(lore_chunks)})
                    print(f"{prefix} 📘 {Col.YELLOW}PDF-LORE{Col.END}   | {rel[:40]:<40} | {Col.YELLOW}Descriptive content{Col.END}")

        # --- 3. TEXT/MD PROCESSING ---
        elif ext in {".md", ".txt"}:
            raw_text = path.read_text(encoding="utf-8", errors="replace")
            content_type = identify_content_type(raw_text, path)
            
            if content_type == "chat":
                convs = parse_chat_text(raw_text, rel) # (Assuming parse_chat_text is defined)
                processed_data["conversations"].extend(convs)
                print(f"{prefix} 📝 {Col.CYAN}TXT-CHAT{Col.END}   | {rel[:40]:<40} | {get_stats_string('chat', len(convs), 'convs')}")
            else:
                processed_data["lore"].append({"source": rel, "text": raw_text})
                word_count = len(raw_text.split())
                print(f"{prefix} 📝 {Col.YELLOW}TXT-LORE{Col.END}   | {rel[:40]:<40} | {Col.YELLOW}{word_count} words{Col.END}")

    except Exception as exc:
        print(f"{prefix} ❌ {Col.RED}FAILED{Col.END}     | {rel[:40]:<40} | Error: {str(exc)[:30]}...")
        processed_data["failed"].append(rel)

elapsed_time = time.time() - start_time

# --- FINAL SMART SUMMARY ---
total_convs = len(processed_data['conversations'])
total_msgs = sum(len(c.get("messages", [])) for c in processed_data["conversations"])
total_lore = len(processed_data['lore'])

print(f"\n{Col.BOLD}═" * 60)
print(f"📊 DATASET ARCHITECT REPORT")
print(f"═" * 60 + Col.END)
print(f"⏱️  Processing Time:   {Col.BOLD}{elapsed_time:.2f} seconds{Col.END}")
print(f"💬 Total Dialogues:    {Col.GREEN}{total_convs:,}{Col.END}")
print(f"💬 Total Messages:     {Col.GREEN}{total_msgs:,}{Col.END}")
print(f"📜 Lore Entries:       {Col.YELLOW}{total_lore:,}{Col.END}")
print(f"❌ Failed Files:       {Col.RED}{len(processed_data['failed'])}{Col.END}")

# Breakdown by source type
if total_convs > 0:
    avg = total_msgs / total_convs
    print(f"📈 Avg Conversation:   {Col.CYAN}{avg:.1f} messages/conv{Col.END}")

print(f"{Col.BOLD}═" * 60 + Col.END)


🧠 Intelligence Engine: Analyzing 196 files...

[1/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_1.md    | Error: name 'is_lore_file' is not def...
[2/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_10.md   | Error: name 'is_lore_file' is not def...
[3/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_11.md   | Error: name 'is_lore_file' is not def...
[4/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_12.md   | Error: name 'is_lore_file' is not def...
[5/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_13.md   | Error: name 'is_lore_file' is not def...
[6/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_14.md   | Error: name 'is_lore_file' is not def...
[7/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_15.md   | Error: name 'is_lore_file' is not def...
[8/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_16.md   | Error: name 'is_lore_file' is not def...
[9/196] ❌ FAILED     | set 1\md files\chats\Nayari_chat_17.md   | Error: name 'is_lore_file' is 

## 4 — Lore → Training Conversations

Each PDF lore section is chunked into paragraphs.
Each chunk is paired with a varied natural-language prompt so the model learns
to recall Nayari's lore organically. Multiple prompts per chunk for augmentation.

In [13]:
import random
import re

# Categories of prompts to make the model more versatile
PROMPT_TEMPLATES = {
    "identity": [
        "Who are you?", "Tell me about yourself.", "Describe your background.",
        "What is your story?", "How would you define your identity?"
    ],
    "knowledge": [
        "Tell me about {subject}.", "What can you share regarding {subject}?",
        "Explain the significance of {subject}.", "Provide some context on {subject}.",
        "What do the records say about {subject}?"
    ],
    "general": [
        "Tell me more.", "Share an entry from the lore.", 
        "What else is there to know?", "Continue the history."
    ]
}

def clean_pdf_text(text):
    """Cleans common PDF extraction artifacts."""
    # Fix hyphenated words at line breaks (e.g., "en- \n vironment" -> "environment")
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # Replace multiple newlines with a standard double newline
    text = re.sub(r'\n\s*\n', '\n\n', text)
    # Replace single newlines with spaces (common in PDF columns)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    return text.strip()

def extract_subject(text):
    """Heuristic to find the 'Subject' of a chunk for better prompting."""
    # Try to grab the first capitalized phrase or the first sentence's noun
    match = re.search(r'([A-Z][a-z]+(?:\s[A-Z][a-z]+)*)', text)
    if match:
        return match.group(1)
    return None

def lore_to_convs_smart(sections, min_chars=200, max_chars=1000, augment=False):
    convs = []
    
    for section in sections:
        # 1. Clean the text
        raw_text = clean_pdf_text(section["text"])
        source = section.get("source", "Unknown Source")
        
        # 2. Smart Chunking (Recursive-ish approach)
        paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
        chunks = []
        current_chunk = ""
        
        for para in paragraphs:
            if len(current_chunk) + len(para) < max_chars:
                current_chunk += ("\n\n" + para if current_chunk else para)
            else:
                if len(current_chunk) >= min_chars:
                    chunks.append(current_chunk)
                current_chunk = para
        if current_chunk:
            chunks.append(current_chunk)

        # 3. Generate Conversations
        for chunk in chunks:
            # Determine subject for dynamic prompting
            subject = extract_subject(chunk)
            
            # Select prompt type
            if "I am" in chunk[:50] or "My name" in chunk[:50]:
                templates = PROMPT_TEMPLATES["identity"]
            elif subject and len(subject) > 3:
                templates = [t.format(subject=subject) for t in PROMPT_TEMPLATES["knowledge"]]
            else:
                templates = PROMPT_TEMPLATES["general"]

            # Augmentation logic
            num_variations = 2 if augment else 1
            selected_prompts = random.sample(templates, min(num_variations, len(templates)))

            for prompt in selected_prompts:
                convs.append({
                    "messages": [
                        {"role": "user", "content": prompt},
                        {"role": "assistant", "content": chunk},
                    ],
                    "metadata": {
                        "source": source,
                        "subject": subject,
                        "length": len(chunk)
                    }
                })
                
    return convs

# Example Usage:
# lore_sections = [{"text": "The Kingdom of Eldoria was founded in... ", "source": "History_Book.pdf"}]
# smart_convs = lore_to_convs_smart(lore_sections, augment=True)

## 5 — Deduplication & Quality Filter

In [16]:
all_raw = processed_data['conversations']
md_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.md')]
txt_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.txt')]
pdf_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.pdf')]
json_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.json')]

all_conversations = all_raw + lore_conversations

before = len(all_conversations)
all_conversations = deduplicate(all_conversations)
after  = len(all_conversations)

print(f"Before dedup : {before}")
print(f"After  dedup : {after}  ({before - after} duplicates removed)")

# Sanity-check: flag any conversation with a very long single message
WARN_CHARS = 3000
flagged = [
    (i+1, m["role"], len(m["content"]))
    for i, c in enumerate(all_conversations)
    for m in c["messages"]
    if len(m["content"]) > WARN_CHARS
]
if flagged:
    print(f"\n⚠ {len(flagged)} message(s) exceed {WARN_CHARS} chars (may be lore, review if unexpected):")
    for idx, role, length in flagged[:8]:
        print(f"  Conv {idx} [{role}]: {length} chars")
else:
    print("\n✅ No oversized messages.")


Before dedup : 99
After  dedup : 99  (0 duplicates removed)

✅ No oversized messages.


## 6 — Preview & Statistics Dashboard

In [17]:
all_raw = processed_data['conversations']
md_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.md')]
txt_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.txt')]
pdf_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.pdf')]
json_conversations = [c for c in all_raw if c.get('source', '').lower().endswith('.json')]

all_conversations = all_raw + lore_conversations

before = len(all_conversations)
all_conversations = deduplicate(all_conversations)
after  = len(all_conversations)

print(f"Before dedup : {before}")
print(f"After  dedup : {after}  ({before - after} duplicates removed)")

# Sanity-check: flag any conversation with a very long single message
WARN_CHARS = 3000
flagged = [
    (i+1, m["role"], len(m["content"]))
    for i, c in enumerate(all_conversations)
    for m in c["messages"]
    if len(m["content"]) > WARN_CHARS
]
if flagged:
    print(f"\n⚠ {len(flagged)} message(s) exceed {WARN_CHARS} chars (may be lore, review if unexpected):")
    for idx, role, length in flagged[:8]:
        print(f"  Conv {idx} [{role}]: {length} chars")
else:
    print("\n✅ No oversized messages.")


Before dedup : 99
After  dedup : 99  (0 duplicates removed)

✅ No oversized messages.


## 7 — Export JSON

In [15]:
from datetime import datetime, timezone

dataset_json = {
    "meta": {
        "version": 3,
        "built_at": datetime.now(timezone.utc).isoformat(),
        "total_conversations": len(all_conversations),
        "sources": {
            "markdown": len(md_conversations),
            "txt":      len(txt_conversations),
            "pdf":      len(pdf_conversations),
            "json":     len(json_conversations),
            "lore":     len(lore_conversations),
        },
    },
    "character": {
        "name": char_name, "type": char_type, "gender": char_gender,
        "traits": char_traits, "personality": char_personality,
        "lore_sections": lore_sections,
    },
    "conversations": all_conversations,
}

OUTPUT_FILE.write_text(
    json.dumps(dataset_json, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
size_kb = OUTPUT_FILE.stat().st_size / 1024
print(f"✅ Saved → {OUTPUT_FILE.resolve()}")
print(f"   {len(all_conversations)} conversations | {size_kb:.1f} KB")


✅ Saved → T:\Documents\Github Desktop\Nayari-AI\nayari_dataset.json
   99 conversations | 222.4 KB


## 8 — Upload to Kaggle via API

**Before running:**
1. Go to [kaggle.com](https://kaggle.com) → Profile → **Settings** → **API** → **Create New Token**
2. A `kaggle.json` downloads — open it and copy the `username` and `key` values
3. Paste them into the two variables below

> The dataset is created as **private** by default.


In [ ]:
# ── FILL THESE IN ──────────────────────────────────────────────────────────
KAGGLE_USERNAME = "kaggle_username"   # from kaggle.json
KAGGLE_KEY      = "kaggle_key"        # from kaggle.json
DATASET_TITLE   = "nayari-dataset"   # slug used in the Kaggle URL
IS_PUBLIC       = False               # True to make the dataset public
# ───────────────────────────────────────────────────────────────────────────

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_file = kaggle_dir / "kaggle.json"
creds_file.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
creds_file.chmod(0o600)
print("✅ Credentials written.")

STAGING = Path("_kaggle_upload")
shutil.rmtree(STAGING, ignore_errors=True)
STAGING.mkdir()
shutil.copy(OUTPUT_FILE, STAGING / OUTPUT_FILE.name)

(STAGING / "dataset-metadata.json").write_text(
    json.dumps({
        "title":     DATASET_TITLE,
        "id":        f"{KAGGLE_USERNAME}/{DATASET_TITLE}",
        "licenses":  [{"name": "CC0-1.0"}],
        "isPrivate": not IS_PUBLIC,
    }, indent=2),
    encoding="utf-8",
)
print(f"✅ Staging folder ready: {STAGING.resolve()}")

def run_kaggle(*args):
    return subprocess.run(["kaggle", *args], capture_output=True, text=True)

result  = run_kaggle("datasets", "create", "-p", str(STAGING), "--dir-mode", "zip")
combined = (result.stdout + result.stderr).lower()

if "already exists" in combined or "403" in combined:
    print("Dataset already exists — pushing a new version...")
    result = run_kaggle(
        "datasets", "version", "-p", str(STAGING),
        "-m", f"v3 build — {len(all_conversations)} conversations",
        "--dir-mode", "zip",
    )

print(result.stdout or result.stderr)

if result.returncode == 0:
    vis = "public" if IS_PUBLIC else "private"
    url = f"https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_TITLE}"
    print(f"🎉 Upload successful! ({vis})")
    print(f"   Dataset URL: {url}")
    print()
    print("📋 Next steps in your Kaggle training notebook:")
    print("   1. Open nayari_train.ipynb on Kaggle")
    print(f"   2. Click '+ Add Data' → search '{DATASET_TITLE}' → Add")
    print("   3. Set Accelerator = GPU T4 x2, Internet = On → Run All")
else:
    print("❌ Upload failed. Double-check KAGGLE_USERNAME and KAGGLE_KEY.")

shutil.rmtree(STAGING, ignore_errors=True)
